In [2]:
import os
os.chdir("/Users/alimurad/Desktop/projects/rag-pgvector/rag_pgvector")
os.listdir()

['.DS_Store', '__init__.py', 'exps', 'src']

In [3]:
from src.utils.utils import notebook_line_magic
notebook_line_magic()

## Database Connection

In [4]:
from dotenv import load_dotenv

import pandas as pd
from src.utils.db_utils import query_db, create_embedding_table, load_embeddings

load_dotenv()

True

In [5]:
from src.data.processing import load_pdf

pages = load_pdf("src/data/cava10k.pdf")
print(f"Loaded {len(pages)} pages")
table_pages = [p["page"] for p in pages if p["has_table"]]
print(f"Pages with tables: {table_pages}")

Loaded 98 pages
Pages with tables: [1, 12, 44, 46, 47, 49, 50, 51, 52, 53, 54, 60, 61, 62, 63, 64, 65, 67, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 83, 93, 94]


In [6]:
# print(pages[45]['table_content'])

from src.utils.utils import get_token_provider
from src.models.embeddings import generate_embeddings
from src.models.chat_completion import chat_completion

token_provider = get_token_provider()
# response = chat_completion(token_provider, "Summarize this: " + pages[45]['table_content'])
# print(response)

### Chunking

In [8]:
pages[0].keys()

dict_keys(['page', 'has_table', 'table_content', 'text'])

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdfplumber import page


def chunk_text(text: str, chunk_size: int = 512, chunk_overlap: int = 64) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_text(text)


chunks = [
    {   
        "page": page['page'],
        "chunks": chunk_text(page['text'])
    } for page in pages
]
temp_chunks = chunks[0]
embeddings = generate_embeddings(
    token_provider=token_provider, 
    texts=temp_chunks['chunks'], 
    page=temp_chunks['page'], 
    dimensions=786
)

In [ ]:
# pd.DataFrame(embeddings)

,id,page,text,timestamp,embedding
0,ac097f65-774d-45e1-803a-b46d87eac276,1,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...,2026-05-01T15:54:42Z,"[-0.023406982421875, 0.0263824462890625, -0.01..."
1,93a52a89-673f-4dc9-b6f6-3f7db333e247,1,Delaware 47-3426661\n(State or other jurisdict...,2026-05-01T15:54:42Z,"[0.040130615234375, 0.002895355224609375, -0.0..."
2,f7eb0fe0-a854-4e01-b462-7f970f24c559,1,Indicate by check mark if the registrant is a ...,2026-05-01T15:54:42Z,"[0.004180908203125, 0.0017147064208984375, -0...."
3,9a20c22f-8345-4fa6-aa8f-0aba9ad79f6f,1,months (or for such shorter period that the re...,2026-05-01T15:54:42Z,"[-0.039886474609375, 0.017547607421875, -0.015..."
4,f799803d-3c42-4eb3-bf4f-bcee059451fe,1,Indicate by check mark whether the registrant ...,2026-05-01T15:54:42Z,"[-0.016510009765625, -0.007415771484375, -0.02..."
5,c6f40369-eafa-4a4f-93f2-fb29cd2f2b93,1,o\nEmerging growth company\nIf an emerging gro...,2026-05-01T15:54:42Z,"[-0.04327392578125, -0.0022487640380859375, -0..."
6,356365bf-df56-4891-816c-ef2929cd64be,1,reporting under Section 404(b) of the Sarbanes...,2026-05-01T15:54:42Z,"[0.0297393798828125, 0.0005917549133300781, -0..."
7,0499e8f1-e5e6-46b4-8490-64d25bb8c6d0,1,Indicate by check mark whether any of those er...,2026-05-01T15:54:42Z,"[-0.0330810546875, -0.0261077880859375, -0.011..."
8,9ac9dfab-84f3-41db-8a8b-19cc85177372,1,"As of July 11, 2025, the last trading day of t...",2026-05-01T15:54:42Z,"[-0.02569580078125, -0.024993896484375, -0.008..."
9,4b475add-bb4f-4832-bf6c-718ce96356ea,1,registrant’s executive officers and directors ...,2026-05-01T15:54:42Z,"[0.035186767578125, -0.01277923583984375, -0.0..."


In [12]:
create_embedding_table(
    schema=os.getenv("PG_DBNAME"),
    table="embeddings",
    embedding_dim=len(embeddings[0]['embedding'])
)

In [ ]:
load_embeddings(
    schema=os.getenv("PG_DBNAME"),
    table="embeddings",
    df=pd.DataFrame(embeddings)
)

10